In [1]:
import pandas as pd 
import xgboost as xgb

In [2]:
train = pd.read_csv('../data/training_data_train_final.csv')
validation = pd.read_csv('../data/training_data_val_final.csv')
test = pd.read_csv('../data/training_data_test_final.csv')

In [3]:
N_SLOTS = 6
df_games = pd.read_csv('../new_package/03_features/game_features.csv')  
GAME_FEATS = [c for c in df_games.columns if c not in ['appid', 'name', 'inclusion', 'status', 'note']]

FEATURE_COLS = (
    [f'slot{i}_{f}'      for i in range(1, N_SLOTS + 1) for f in GAME_FEATS] +
    [f'slot{i}_interest' for i in range(1, N_SLOTS + 1)] +
    [f'slot{i}_present'  for i in range(1, N_SLOTS + 1)] +
    [f'cand_{f}'         for f in GAME_FEATS]
)
dtrain = xgb.DMatrix(train[FEATURE_COLS],   label=train['interest'])
dval   = xgb.DMatrix(validation[FEATURE_COLS],   label=validation['interest'])
dtest  = xgb.DMatrix(test[FEATURE_COLS], label=test['interest'])

In [4]:
import os, json
import xgboost as xgb

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH  = os.path.join(MODEL_DIR, "model_slot_deep.json")
BUNDLE_PATH = os.path.join(MODEL_DIR, "model_slot_deep_bundle.json")

params = {
    'objective': 'reg:squarederror',
    'eval_metric': ['rmse', 'mae'],
    'tree_method': 'hist',
    'eta': 0.03,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.5,
    'min_child_weight': 5,
    'reg_lambda': 1.0,
    'seed': 0,
}

if os.path.exists(MODEL_PATH) and os.path.exists(BUNDLE_PATH):
    # ---- load ----
    model = xgb.Booster()
    model.load_model(MODEL_PATH)

    with open(BUNDLE_PATH) as f:
        bundle = json.load(f)
    FEATURE_COLS = bundle['feature_cols']
    best_iter = bundle['best_iteration']
    print(f"Loaded model from {MODEL_PATH} (best_iteration={best_iter})")
else:
    # ---- train ----
    model = xgb.train(
        params, dtrain,
        num_boost_round=3000,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=50,
        verbose_eval=100,
    )
    best_iter = model.best_iteration

    # ---- save model + bundle ----
    model.save_model(MODEL_PATH)

    bundle = {
        'feature_cols':   FEATURE_COLS,
        'game_feats':     GAME_FEATS,
        'n_slots':        N_SLOTS,
        'exclude_cols':   ['appid', 'name', 'inclusion', 'status', 'note'],
        'params':         params,
        'best_iteration': int(best_iter),
    }
    with open(BUNDLE_PATH, 'w') as f:
        json.dump(bundle, f)
    print(f"Trained and saved model to {MODEL_PATH} (best_iteration={best_iter})")


Loaded model from ../models/model_slot_deep.json (best_iteration=2999)


In [5]:
import numpy as np

def test_model(model):
    grand_mean = train['interest'].mean()
    baseline_rmse = np.sqrt(np.mean((validation['interest'] - grand_mean)**2))
    print('global-mean RMSE:', baseline_rmse)
    # or per-user mean (better baseline)
    user_mean = train.groupby('steamid')['interest'].mean()
    validation_base = validation.assign(pred=validation['steamid'].map(user_mean)).dropna(subset=['pred'])
    print('per-user-mean RMSE:', np.sqrt(np.mean((validation_base['interest'] - validation_base['pred'])**2)))
    from scipy.stats import spearmanr
    pred = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
    validation_pred = validation.assign(pred=pred)
    sp = (validation_pred.groupby('steamid')
            .filter(lambda g: len(g) >= 2 and g['interest'].nunique() > 1)
            .groupby('steamid')
            .apply(lambda g: spearmanr(g['interest'], g['pred']).correlation))
    print('mean Spearman:', sp.mean(), '| users:', sp.notna().sum())

In [6]:
test_model(model)

global-mean RMSE: 0.17448781254161982
per-user-mean RMSE: 0.16442796141650518
mean Spearman: 0.2807209984788752 | users: 19892


/tmp/ipykernel_3112120/438608121.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: spearmanr(g['interest'], g['pred']).correlation))


In [7]:
import os
import xgboost as xgb
import json

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH  = os.path.join(MODEL_DIR, "model2_slot_reg.json")
BUNDLE_PATH = os.path.join(MODEL_DIR, "model2_slot_reg_bundle.json")

params2 = {
    'objective': 'reg:squarederror',
    'eval_metric': ['rmse', 'mae'],
    'tree_method': 'hist',
    'eta': 0.05,
    'max_depth': 4,
    'subsample': 0.7,
    'colsample_bytree': 0.3,
    'min_child_weight': 20,
    'reg_lambda': 5.0,
    'reg_alpha': 1.0,
    'seed': 0,
}

if os.path.exists(MODEL_PATH) and os.path.exists(BUNDLE_PATH):
    # ---- load ----
    model2 = xgb.Booster()
    model2.load_model(MODEL_PATH)

    with open(BUNDLE_PATH) as f:
        bundle = json.load(f)
    FEATURE_COLS = bundle['feature_cols']
    best_iter = bundle['best_iteration']
    print(f"Loaded model2 from {MODEL_PATH} (best_iteration={best_iter})")
else:
    # ---- train ----
    model2 = xgb.train(
        params2, dtrain,
        num_boost_round=3000,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=50,
        verbose_eval=100,
    )
    best_iter = model2.best_iteration

    # ---- save model + bundle ----
    model2.save_model(MODEL_PATH)

    bundle = {
        'feature_cols':   FEATURE_COLS,
        'game_feats':     GAME_FEATS,
        'n_slots':        N_SLOTS,
        'exclude_cols':   ['appid', 'name', 'inclusion', 'status', 'note'],
        'params':         params2,
        'best_iteration': int(best_iter),
    }
    with open(BUNDLE_PATH, 'w') as f:
        json.dump(bundle, f)
    print(f"Trained and saved model2 to {MODEL_PATH} (best_iteration={best_iter})")

Loaded model2 from ../models/model2_slot_reg.json (best_iteration=2969)


In [8]:
test_model(model2)

global-mean RMSE: 0.17448781254161982
per-user-mean RMSE: 0.16442796141650518
mean Spearman: 0.2758965722996975 | users: 19892


/tmp/ipykernel_3112120/438608121.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: spearmanr(g['interest'], g['pred']).correlation))


In [9]:
# Test against cand-only baseline, without the user's other games
import os, json
import xgboost as xgb
from scipy.stats import spearmanr

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH  = os.path.join(MODEL_DIR, "model_cand_only.json")
BUNDLE_PATH = os.path.join(MODEL_DIR, "model_cand_only_bundle.json")

CAND_ONLY = [f'cand_{f}' for f in GAME_FEATS]

if os.path.exists(MODEL_PATH) and os.path.exists(BUNDLE_PATH):
    # ---- load ----
    base = xgb.Booster()
    base.load_model(MODEL_PATH)

    with open(BUNDLE_PATH) as f:
        bundle = json.load(f)
    CAND_ONLY = bundle['feature_cols']
    best_iter = bundle['best_iteration']
    print(f"Loaded base from {MODEL_PATH} (best_iteration={best_iter})")
else:
    # ---- train ----
    dtr_b = xgb.DMatrix(train[CAND_ONLY],      label=train['interest'])
    dva_b = xgb.DMatrix(validation[CAND_ONLY], label=validation['interest'])

    base = xgb.train(
        params2, dtr_b,
        num_boost_round=3000,
        evals=[(dtr_b, 'train'), (dva_b, 'val')],
        early_stopping_rounds=50,
        verbose_eval=False,
    )
    best_iter = base.best_iteration

    # ---- save ----
    base.save_model(MODEL_PATH)
    bundle = {
        'feature_cols':   CAND_ONLY,
        'game_feats':     GAME_FEATS,
        'n_slots':        None,
        'exclude_cols':   ['appid', 'name', 'inclusion', 'status', 'note'],
        'params':         params2,
        'best_iteration': int(best_iter),
    }
    with open(BUNDLE_PATH, 'w') as f:
        json.dump(bundle, f)
    print(f"Trained and saved base to {MODEL_PATH} (best_iteration={best_iter})")

# ---- evaluate ----
dva_b = xgb.DMatrix(validation[CAND_ONLY], label=validation['interest'])
pred_b = base.predict(dva_b, iteration_range=(0, best_iter + 1))
va_b = validation.assign(pred=pred_b)
sp_b = (va_b.groupby('steamid')
             .filter(lambda g: len(g) >= 2 and g['interest'].nunique() > 1)
             .groupby('steamid')
             .apply(lambda g: spearmanr(g['interest'], g['pred']).correlation))
print('candidate-only Spearman:', sp_b.mean())

Loaded base from ../models/model_cand_only.json (best_iteration=1701)
candidate-only Spearman: 0.25974802343815917


/tmp/ipykernel_3112120/1406760922.py:59: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: spearmanr(g['interest'], g['pred']).correlation))


In [10]:
importance = model.get_score(importance_type='gain')
cand_gain = sum(v for k,v in importance.items() if k.startswith('cand_'))
slot_gain = sum(v for k,v in importance.items() if k.startswith('slot'))
print('candidate gain:', cand_gain)
print('slot gain:     ', slot_gain)
print('slot share:    ', slot_gain / (cand_gain + slot_gain))

candidate gain: 23.479016162455082
slot gain:      66.44987114705145
slot share:     0.738915749266998


In [11]:
importance = model2.get_score(importance_type='gain')
cand_gain = sum(v for k,v in importance.items() if k.startswith('cand_'))
slot_gain = sum(v for k,v in importance.items() if k.startswith('slot'))
print('candidate gain:', cand_gain)
print('slot gain:     ', slot_gain)
print('slot share:    ', slot_gain / (cand_gain + slot_gain))

candidate gain: 23.442725982517004
slot gain:      67.48104843497276
slot share:     0.7421716582631479


In [12]:
pred_slot = model.predict(dval)
pred_cand = base.predict(dva_b)


# correlation between the two models' predictions
print('corr(slot_pred, cand_pred):', np.corrcoef(pred_slot, pred_cand)[0,1])

corr(slot_pred, cand_pred): 0.601233396889575


In [13]:
va2 = validation.assign(pred_slot=pred_slot, pred_cand=pred_cand)

# per-user Spearman for each model
def per_user_sp(g, col):
    if len(g) < 2 or g['interest'].nunique() < 2:
        return np.nan
    return spearmanr(g['interest'], g[col]).correlation

sp = (va2.groupby('steamid')
        .apply(lambda g: pd.Series({
            'slot': per_user_sp(g, 'pred_slot'),
            'cand': per_user_sp(g, 'pred_cand')})))
print(sp.mean())
print('users where slot beats cand:', (sp['slot'] > sp['cand']).mean())

slot    0.280721
cand    0.259324
dtype: float64
users where slot beats cand: 0.2604899466147782


/tmp/ipykernel_3112120/2781445841.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
